In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import importlib, qwen_from_scratch
importlib.reload(qwen_from_scratch)
from qwen_from_scratch import MyQwen, RMSNorm, AttentionProjections, RotaryEmbedding, repeat_kv as repeat_kv_my, MLP, DecoderLayer, QwenModel
import inspect
import torch.nn.functional as F


c:\Users\pra19\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# Load Hugging Face model
hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # torch_dtype="auto",
    torch_dtype=torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("=== CONFIG ===")
print(hf_model.config)

print("\n=== MODEL ===")
print(hf_model)

print("\n=== WEIGHTS ===")
for name, param in hf_model.named_parameters():
    print(name, tuple(param.shape))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 627.34it/s]


=== CONFIG ===
Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float32",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_head

In [4]:
print(inspect.getsource(type(
    hf_model.model.layers[0].self_attn
)))

@use_kernelized_func(apply_rotary_pos_emb)
class Qwen2Attention(nn.Module):
    """Multi-headed attention from 'Attention Is All You Need' paper"""

    def __init__(self, config: Qwen2Config, layer_idx: int):
        super().__init__()
        self.layer_type = config.layer_types[layer_idx] if hasattr(config, "layer_types") else None
        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
        self.num_key_value_groups = config.num_attention_heads // config.num_key_value_heads
        self.scaling = self.head_dim**-0.5
        self.attention_dropout = config.attention_dropout
        self.is_causal = True
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=True)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=True)
        self.v_proj = nn.Linear(config.hidden_size, config.

In [5]:
print(hf_model.model.rotary_emb)
print(inspect.getsource(type(hf_model.model.rotary_emb)))
print(inspect.getsource(type(hf_model.model.layers[0].self_attn).forward))

Qwen2RotaryEmbedding()
class Qwen2RotaryEmbedding(nn.Module):
    @deprecate_kwarg("device", version="5.18")
    def __init__(self, config: Qwen2Config, device=None):
        super().__init__()
        self.max_seq_len_cached = config.max_position_embeddings
        self.original_max_seq_len = config.max_position_embeddings

        self.config = config

        self.rope_type = self.config.rope_parameters["rope_type"]
        rope_init_fn: Callable = self.compute_default_rope_parameters
        if self.rope_type != "default":
            rope_init_fn = ROPE_INIT_FUNCTIONS[self.rope_type]
        inv_freq, self.attention_scaling = rope_init_fn(self.config, device)

        self.inv_freq = nn.Buffer(inv_freq, persistent=False)
        self.original_inv_freq = nn.Buffer(inv_freq.clone(), persistent=False)

    @staticmethod
    @deprecate_kwarg("device", version="5.18")
    def compute_default_rope_parameters(config: Qwen2Config, device=None, **kwargs) -> tuple[torch.Tensor, float]:
    

In [23]:
# Create our model
my_model = QwenModel(hf_model.config)
my_model.to(hf_model.device)
# Copy Qwen's embedding weights into our embedding layer
with torch.no_grad():
    my_model.embed_tokens.weight.copy_(
        hf_model.model.embed_tokens.weight
    )

In [44]:
# Convert text into token IDs
input_ids = tokenizer(
    "Hello, how are you?",
    return_tensors="pt"
).input_ids.to(hf_model.device)

# Create some test data
x = torch.randn(
    4,                              # batch size
    64,                              # sequence length
    hf_model.config.hidden_size     # hidden dimension
).to(hf_model.device)


# Testing embeddings

In [7]:
# HF embedding output
with torch.no_grad():
    hf_output = hf_model.model.embed_tokens(input_ids)
# Our embedding output
with torch.no_grad():
    my_output = my_model.embed_tokens(input_ids)    

print("HF embedding output shape:", hf_output.shape, my_output.dtype,"\n", hf_output)
print("\nOur embedding output shape:", my_output.shape, my_output.dtype,"\n", my_output)   

# Compare
max_diff = (hf_output - my_output).abs().max().item()

print("Input IDs shape:", input_ids.shape)
print("HF output shape:", hf_output.shape)
print("My output shape:", my_output.shape)
print("Max difference:", max_diff)

print(
    "Match:",
    torch.allclose(
        hf_output,  
        my_output)
)

HF embedding output shape: torch.Size([1, 6, 896]) torch.float32 
 tensor([[[-0.0258,  0.0052, -0.0104,  ...,  0.0027, -0.0085,  0.0039],
         [-0.0222, -0.0136, -0.0153,  ...,  0.0165,  0.0056, -0.0289],
         [-0.0007, -0.0145, -0.0060,  ..., -0.0043,  0.0065, -0.0229],
         [-0.0029,  0.0210, -0.0023,  ...,  0.0181,  0.0164, -0.0082],
         [ 0.0026, -0.0258,  0.0094,  ...,  0.0251, -0.0002, -0.0183],
         [-0.0175,  0.0119, -0.0086,  ...,  0.0211,  0.0022, -0.0131]]],
       device='cuda:0')

Our embedding output shape: torch.Size([1, 6, 896]) torch.float32 
 tensor([[[-0.0258,  0.0052, -0.0104,  ...,  0.0027, -0.0085,  0.0039],
         [-0.0222, -0.0136, -0.0153,  ...,  0.0165,  0.0056, -0.0289],
         [-0.0007, -0.0145, -0.0060,  ..., -0.0043,  0.0065, -0.0229],
         [-0.0029,  0.0210, -0.0023,  ...,  0.0181,  0.0164, -0.0082],
         [ 0.0026, -0.0258,  0.0094,  ...,  0.0251, -0.0002, -0.0183],
         [-0.0175,  0.0119, -0.0086,  ...,  0.0211,  0.00

# Testing RMSNorm

In [ ]:
# Get Qwen's first RMSNorm layer
hf_norm = hf_model.model.layers[0].input_layernorm

# Create our RMSNorm
my_norm = RMSNorm(hf_model.config).to(hf_model.device)

# Copy the pretrained RMSNorm weights
with torch.no_grad():
    my_norm.weight.copy_(hf_norm.weight)


# Run both implementations
with torch.no_grad():
    hf_output = hf_norm(x)
    my_output = my_norm(x)


# Compare
max_diff = (
    hf_output - my_output
).abs().max().item()


print("Input shape:", x.shape)
print("HF output shape:", hf_output.shape)
print("My output shape:", my_output.shape)

print("\nMax difference:", max_diff)

print(
    "Match:",
    torch.allclose(
        hf_output,
        my_output)
)

Input shape: torch.Size([4, 64, 896])
HF output shape: torch.Size([4, 64, 896])
My output shape: torch.Size([4, 64, 896])

Max difference: 2.384185791015625e-07
Match: True


# Q, K, V projections verification

In [30]:
hf_layer = hf_model.model.layers[0]

my_attention = AttentionProjections(hf_model.config, debug=True).to(hf_model.device)

with torch.no_grad():
    my_attention.q_proj.weight.copy_(
        hf_layer.self_attn.q_proj.weight
    )
    my_attention.q_proj.bias.copy_(
        hf_layer.self_attn.q_proj.bias
    )
    my_attention.k_proj.weight.copy_(
    hf_layer.self_attn.k_proj.weight
    )
    my_attention.k_proj.bias.copy_(
    hf_layer.self_attn.k_proj.bias
    )
    my_attention.v_proj.weight.copy_(
    hf_layer.self_attn.v_proj.weight
    )
    my_attention.v_proj.bias.copy_(
    hf_layer.self_attn.v_proj.bias
    )
    # o_proj was missing -- without it the output goes through random weights
    my_attention.o_proj.weight.copy_(
    hf_layer.self_attn.o_proj.weight
    )

# Create some test data
x = torch.randn(
    4,                              # batch size
    64,                              # sequence length
    hf_model.config.hidden_size     # hidden dimension
).to(hf_model.device)

print("Input shape:", x.shape)

# Compare Q projections
hf_q = hf_layer.self_attn.q_proj(x)
my_q = my_attention.q_proj(x)
q_diff = (hf_q - my_q).abs().max().item()
print("Max difference in Q projection:", q_diff)
print("Match:",torch.allclose(hf_q, my_q))

# Compare K projections
hf_k = hf_layer.self_attn.k_proj(x)
my_k = my_attention.k_proj(x)
k_diff = (hf_k - my_k).abs().max().item()
print("Max difference in K projection:", k_diff)
print("Match:",torch.allclose(hf_k, my_k))

# Compare V projections
hf_v = hf_layer.self_attn.v_proj(x)
my_v = my_attention.v_proj(x)
v_diff = (hf_v - my_v).abs().max().item()
print("Max difference in V projection:", v_diff)
print("Match:",torch.allclose(hf_v, my_v))

Input shape: torch.Size([4, 64, 896])
Max difference in Q projection: 0.0
Match: True
Max difference in K projection: 0.0
Match: True
Max difference in V projection: 0.0
Match: True


# Q, K, V shapes verification after reshaping for heads

In [12]:
# Run Qwen's official Query projection.
hf_q = hf_layer.self_attn.q_proj(x)

# Get the input dimensions.
batch_size, sequence_length, _ = x.shape

# Reshape Q exactly according to Qwen's attention configuration.
hf_q = hf_q.view(
    batch_size,
    sequence_length,
    hf_model.config.num_attention_heads,
    hf_model.config.hidden_size //
    hf_model.config.num_attention_heads
).transpose(1, 2)
position_ids = torch.arange(
    x.shape[1],
    device=x.device
).unsqueeze(0)

my_q, my_k, my_v,_,_,_,_,_,_,_ = my_attention.forward(x, position_ids)
# print*("HF Q shape:", my_attention.forward(x)[0].shape)
print("HF Q shape:", hf_q.shape)
print("My Q shape:", my_q.shape)    
q_diff = (hf_q - my_q).abs().max().item()

print("Q max difference:", q_diff)

# Check whether both tensors match numerically.
print("Q matches:", torch.allclose(hf_q, my_q))

HF Q shape: torch.Size([4, 14, 64, 64])
My Q shape: torch.Size([4, 14, 64, 64])
Q max difference: 42.65734100341797
Q matches: False


# Testing RoPE

In [45]:
position_ids = torch.arange(
    x.shape[1],
    device=x.device
).unsqueeze(0)

# Generate the official Qwen cosine and sine values.
hf_cos, hf_sin = hf_model.model.rotary_emb(
    x,
    position_ids
)

my_rotary_emb = RotaryEmbedding(config=hf_model.config).to(hf_model.device)
# Generate cosine and sine values using our implementation.
my_cos, my_sin = my_rotary_emb(
    x,
    position_ids
)

# Check the largest numerical difference.
print(
    "Cos max difference:",
    (hf_cos - my_cos).abs().max().item()
)

print(
    "Sin max difference:",
    (hf_sin - my_sin).abs().max().item()
)

# Final numerical checks.
print(
    "Cos matches:",
    torch.allclose(hf_cos, my_cos, atol=1e-4)
)

print(
    "Sin matches:",
    torch.allclose(hf_sin, my_sin, atol=1e-4)
)

Cos max difference: 0.0
Sin max difference: 0.0
Cos matches: True
Sin matches: True


# verifying RoPE rotation

In [17]:
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb

my_q, my_k, my_v, my_q_before_rope, my_k_before_rope, my_q_post_rope, my_k_post_rope, my_v_post_rope, my_attention_scores_raw, my_attention_output = my_attention(x, position_ids)
hf_cos, hf_sin = hf_model.model.rotary_emb(x, position_ids)
hf_q, hf_k = apply_rotary_pos_emb(
    my_q_before_rope,
    my_k_before_rope,
    hf_cos,
    hf_sin
)

q_diff = (my_q_post_rope - hf_q).abs().max().item()
k_diff = (my_k_post_rope - hf_k).abs().max().item()

print("Q max difference:", q_diff)
print("K max difference:", k_diff)

print(
    "Q matches:",
    torch.allclose(my_q_post_rope, hf_q, atol=1e-4)
)

print(
    "K matches:",
    torch.allclose(my_k_post_rope, hf_k, atol=1e-4)
)

Q max difference: 0.0
K max difference: 0.0
Q matches: True
K matches: True


# Attention output verification

In [18]:

print(
    inspect.getsource(
        type(hf_layer.self_attn).forward
    )
)

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: torch.Tensor | None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[FlashAttentionKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(key_states, value_states, self.layer_idx)

        attention_interface: Callable = AL

In [19]:
print(hf_model.config._attn_implementation)

sdpa


In [20]:
my_q, my_k, my_v, my_q_before_rope, my_k_before_rope, my_q_post_rope, my_k_post_rope, my_v_post_rope, my_attention_scores_raw, my_attention_output = my_attention(
    x,
    position_ids
)
position_embeddings = hf_model.model.rotary_emb(
    x,
    position_ids
)
# Create a causal mask for the current sequence.
#
# Lower triangle = allowed attention.
# Upper triangle = blocked future tokens.
causal_mask = torch.triu(
    torch.full(
        (x.shape[1], x.shape[1]),
        float("-inf"),
        device=x.device,
        dtype=x.dtype
    ),
    diagonal=1
)

# Add batch and head dimensions so the mask has shape:
# [batch_size, 1, sequence_length, sequence_length]
attention_mask = causal_mask.unsqueeze(0).unsqueeze(0)
hf_output, _ = hf_layer.self_attn(
    x,
    position_embeddings=position_embeddings,
    attention_mask=attention_mask
)

# Calculate the largest absolute difference between
# Hugging Face and our complete attention output.
max_difference = (
    hf_output - my_attention_output
).abs().max().item()

print("Max difference:", max_difference)

# Check whether both outputs are numerically equivalent.
print(
    "Outputs match:",
    torch.allclose(
        hf_output,
        my_attention_output,
        atol=1e-4
    )
)

import math
from transformers.models.qwen2.modeling_qwen2 import repeat_kv
num_repeats = my_attention.num_key_value_groups
# Run Hugging Face's repeat_kv on the same K and V tensors.
hf_k = repeat_kv(
    my_k_post_rope,
    num_repeats
)

# Run your implementation.
my_k_repeated = repeat_kv_my(
    my_k_post_rope,
    num_repeats
)
# Compare Key repetition.
print(
    "K max difference:",
    (hf_k - my_k_repeated).abs().max().item()
)

print(
    "K matches:",
    torch.allclose(hf_k, my_k, atol=1e-4)
)

hf_v = repeat_kv(
    my_v_post_rope,
    num_repeats
)
my_v_repeated = repeat_kv_my(
    my_v_post_rope,
    num_repeats
)
# Compare Value repetition.
print(
    "V max difference:",
    (hf_v - my_v_repeated).abs().max().item()
)

print(
    "V matches:",
    torch.allclose(hf_v, my_v_repeated, atol=1e-4)
)


expected_attention_scores = torch.matmul(
    my_q_post_rope,
    my_k_repeated.transpose(-2, -1)
)
# expected_attention_scores = expected_attention_scores / math.sqrt(
#     64
# )
# expected_attention_scores = expected_attention_scores.masked_fill(
#     causal_mask == 0,
#     float("-inf")
# )

# attention_weights = torch.softmax(
#     expected_attention_scores,
#     dim=-1
# )
# expected_attention_scores = torch.matmul(
#     attention_weights,
#     my_v_repeated
# )
# Compare the scores saved inside the attention module
# with the manually computed scores.
print(
    "Raw attention score max difference:",
    (
        my_attention_scores_raw -
        expected_attention_scores
    ).abs().max().item()
)

print(
    "Raw attention scores match:",
    torch.allclose(
        my_attention_scores_raw,
        expected_attention_scores,
        atol=1e-4
    )
)

Max difference: 1.4603137969970703e-05
Outputs match: True
K max difference: 0.0
K matches: True
V max difference: 0.0
V matches: True
Raw attention score max difference: 0.0
Raw attention scores match: True


# Testing MLP

In [21]:
print(hf_model.model.layers[0].mlp)

Qwen2MLP(
  (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
  (up_proj): Linear(in_features=896, out_features=4864, bias=False)
  (down_proj): Linear(in_features=4864, out_features=896, bias=False)
  (act_fn): SiLUActivation()
)


In [22]:
print(inspect.getsource(type(hf_model.model.layers[0].mlp).forward))

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj



In [32]:
hf_mlp = hf_model.model.layers[0].mlp
my_mlp = MLP(hf_model.config).to(hf_model.device)
my_mlp.load_state_dict(hf_mlp.state_dict(), strict=True)   # copies all 3 weights, errors if names differ

x = torch.randn(4, 64, hf_model.config.hidden_size, device=hf_model.device)
with torch.no_grad():
    diff = (hf_mlp(x) - my_mlp(x)).abs().max().item()
print("MLP max diff:", diff)
print("MLP match:", torch.allclose(hf_mlp(x), my_mlp(x)))

MLP max diff: 0.0
MLP match: True


# Testing whole decoder layer

In [38]:
print(hf_model.model.layers[0])

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [37]:
print(inspect.getsource(type(hf_model.model.layers[0]).forward))

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = False,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        # Self Attention
        hidden_states, _ = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        hidden_states = residual + hidden_states

        # Fully Connected
        residual = hidden_states
        hidden_states = self.post_att

In [27]:
hf_layer = hf_model.model.layers[0]
my_layer = DecoderLayer(hf_model.config).to(hf_model.device)
my_layer.load_state_dict(hf_layer.state_dict(), strict=True)   # copies all 3 weights, errors if names differ

x = torch.randn(4, 8, hf_model.config.hidden_size, device=hf_model.device)
position_ids = torch.arange(8, device=x.device).unsqueeze(0)
mask = torch.triu(torch.full((8, 8), float("-inf"), device=x.device), 1)[None, None]
my_rotary_emb = RotaryEmbedding(config=hf_model.config).to(hf_model.device)
# Generate cosine and sine values using our implementation.
with torch.no_grad():
    hf_out = hf_layer(
        x,
        position_embeddings=hf_model.model.rotary_emb(x, position_ids),
        attention_mask=mask,
    )
    if isinstance(hf_out, tuple):     # older transformers versions return a tuple
        hf_out = hf_out[0]
    my_cos, my_sin = my_rotary_emb(x,position_ids)
    my_out = my_layer(x, my_cos, my_sin)

print("Decoder layer max diff:", (hf_out - my_out).abs().max().item())
print("Decoder layer match:", torch.allclose(hf_out, my_out, atol=1e-5))

Decoder layer max diff: 4.76837158203125e-07
Decoder layer match: True


# Finallyyy the whole model

In [28]:
print(hf_model.model)

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)


In [29]:
print(inspect.getsource(type(hf_model.model).forward))

    @merge_with_config_defaults
    @capture_outputs
    @auto_docstring
    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        inputs_embeds: torch.FloatTensor | None = None,
        use_cache: bool | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutputWithPast:
        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config)

        if position_ids is None:
            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
            position_id

In [6]:
my_model = MyQwen(hf_model.config).to(hf_model.device)
missing, unexpected = my_model.model.load_state_dict(hf_model.model.state_dict(), strict=True)
my_model.eval()

MyQwen(
  (model): QwenModel(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x DecoderLayer(
        (self_attn): AttentionProjections(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        )
        (input_layernorm): RMSNorm()
        (post_attention_layernorm): RMSNorm()
      )
    )
    (norm): RMSNorm()
    (rotary_emb): RotaryEmbedding()
  )
)

In [8]:
# Check whether Qwen ties the input embedding and output head weights:
print(
    hf_model.model.embed_tokens.weight.data_ptr()
    ==
    hf_model.lm_head.weight.data_ptr())
print(
    hf_model.model.embed_tokens.weight.shape,
    hf_model.lm_head.weight.shape
)

True
torch.Size([151936, 896]) torch.Size([151936, 896])


In [10]:
input_ids = tokenizer("The capital of France is", return_tensors="pt").input_ids.to(hf_model.device)

with torch.no_grad():
    hf_out = hf_model(input_ids, output_hidden_states=True)
    my_logits, my_hidden = my_model(input_ids, return_hidden=True)

# Per-layer hidden state comparison.
# hf_out.hidden_states[0] is the embedding output, [i] is after layer i-1.
# NOTE: HF applies the final norm to the LAST entry, so compare only 0..23 here.
for i in range(len(my_hidden) - 1):
    d = (hf_out.hidden_states[i] - my_hidden[i]).abs().max().item()
    print(f"hidden[{i:2d}] max diff = {d:.2e}")

print("HF logits shape:", hf_out.logits.shape)
print("My logits shape:", my_logits.shape)

print("logits max diff =", (hf_out.logits - my_logits).abs().max().item())

# The actual milestone check:
print("M1 parity:", torch.allclose(hf_out.logits, my_logits, atol=1e-4))

# Sanity: both models predict the same next token
print("HF next token :", tokenizer.decode(hf_out.logits[0, -1].argmax()))
print("My next token :", tokenizer.decode(my_logits[0, -1].argmax()))

hidden[ 0] max diff = 0.00e+00
hidden[ 1] max diff = 5.48e-06
hidden[ 2] max diff = 3.81e-06
hidden[ 3] max diff = 6.10e-05
hidden[ 4] max diff = 1.22e-04
hidden[ 5] max diff = 1.22e-04
hidden[ 6] max diff = 1.22e-04
hidden[ 7] max diff = 1.22e-04
hidden[ 8] max diff = 1.22e-04
hidden[ 9] max diff = 1.22e-04
hidden[10] max diff = 1.22e-04
hidden[11] max diff = 1.22e-04
hidden[12] max diff = 1.22e-04
hidden[13] max diff = 1.22e-04
hidden[14] max diff = 1.22e-04
hidden[15] max diff = 1.22e-04
hidden[16] max diff = 1.22e-04
hidden[17] max diff = 1.22e-04
hidden[18] max diff = 1.22e-04
hidden[19] max diff = 1.22e-04
hidden[20] max diff = 1.22e-04
hidden[21] max diff = 1.22e-04
hidden[22] max diff = 3.66e-04
hidden[23] max diff = 1.70e-04
HF logits shape: torch.Size([1, 5, 151936])
My logits shape: torch.Size([1, 5, 151936])
logits max diff = 3.337860107421875e-05
M1 parity: True
HF next token :  Paris
My next token :  Paris
